<a href="https://colab.research.google.com/github/gaelBZH/NumericalMethods/blob/main/Laboratory6/Laboratory_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratory 6 🌍
Gaël Kerninon
---

---
### Exercise 2: Gaussian Substitution

$$
\begin{cases}
4x_1 - x_2 + x_3 = 8 & (L_1) \\
2x_1 + 5x_2 + 2x_3 = 3 & (L_2) \\
x_1 + 2x_2 + 4x_3 = 11 & (L_3)
\end{cases}
$$

**Step 1: Eliminate x1**

$$L_1 \leftarrow L_1 - 2L_2$$
$$L_3 \leftarrow L_1 - 4L_3$$

$$
\begin{cases}
-11x_2 - 3x_3 = 2 & (L_1) \\
2x_1 + 5x_2 + 2x_3 = 3 & (L_2) \\
-9x_2 - 15x_3 = -36 & (L_3)
\end{cases}
$$

**Step 2: Simplification and Substitution**
$$L_3 \leftarrow L_3 \div (-3)$$

$$
\begin{cases}
-11x_2 - 3x_3 = 2 & (L_1) \\
2x_1 + 5x_2 + 2x_3 = 3 & (L_2) \\
3x_2 + 5x_3 = 12 \implies x_2 = \frac{12 - 5x_3}{3} & (L_3)
\end{cases}
$$

$$L_1 \leftarrow \text{Substitution of } x_2$$

$$
\begin{cases}
-11\left(\frac{12 - 5x_3}{3}\right) - 3x_3 = 2 \implies 46x_3 = 138 \implies x_3 = 3 & (L_1) \\
2x_1 + 5x_2 + 2x_3 = 3 & (L_2) \\
x_2 = \frac{12 - 5(3)}{3} \implies x_2 = -1 & (L_3)
\end{cases}
$$

**Step 3: Solving**
$$
\begin{cases}
x_3 = 3 & (L_1) \\
2x_1 + 5(-1) + 2(3) = 3 \implies 2x_1 + 1 = 3 \implies x_1 = 1 & (L_2) \\
x_2 = -1 & (L_3)
\end{cases}
$$

**Conclusion:**
$$
\begin{cases}
x_1 = 1 \\
x_2 = -1 \\
x_3 = 3
\end{cases}
$$

---
### Exercise 3: Gaussian Elimination with Pivoting in Python
Pivoting ensures equations are ordered such that coefficients with the greatest absolute value are on the diagonal. We will solve the system from Exercise 2 and compare the results to the library function `scipy.linalg.solve`.

In [4]:
import scipy.linalg
import numpy as np
import copy

def linearsolver(A, b):
    n = len(A)
    M = copy.deepcopy(A)
    b_copy = copy.deepcopy(b)

    i = 0
    for x in M:
        x.append(b_copy[i])
        i += 1

    for k in range(n):
        # Pivoting
        for i in range(k, n):
            if abs(M[i][k]) > abs(M[k][k]):
                M[k], M[i] = M[i], M[k]
            else:
                pass

        # Gaussian elimination
        for j in range(k + 1, n):
            q = float(M[j][k]) / M[k][k]
            for m in range(k, n + 1):
                M[j][m] -= q * M[k][m]

    x = [0 for i in range(n)]

    # Back substitution
    x[n - 1] = float(M[n - 1][n]) / M[n - 1][n - 1]
    for i in range(n - 1, -1, -1):
        z = 0
        for j in range(i + 1, n):
            z = z + float(M[i][j]) * x[j]
        x[i] = float(M[i][n] - z) / M[i][i]

    return x

# System of equations from Ex 2
A = [[4, -1, 1], [2, 5, 2], [1, 2, 4]]
b = [8, 3, 11]

# Solving with custom implementation
x_custom = linearsolver(A, b)
print("Result with custom linearsolver:")
print(x_custom)

# Solving with SciPy library function
x_scipy = scipy.linalg.solve(A, b)
print("\nResult with scipy.linalg.solve:")
print(x_scipy)

Result with custom linearsolver:
[1.0, -1.0, 3.0]

Result with scipy.linalg.solve:
[ 1. -1.  3.]


---
### Exercise 4: LU Decomposition
This section utilizes code that performs LU decomposition, returning lower (L), upper (U), and pivoting (P) matrices.

We will solve the following lecture set:
$$(\begin{matrix}1&2&1\\ 1&-2&2\\ 2&12&-2\end{matrix})(\begin{matrix}x_{1}\\ x_{2}\\ x_{3}\end{matrix})=(\begin{matrix}0\\ 4\\ 4\end{matrix})$$

In [5]:
import numpy as np
import scipy.linalg

def LU_partial_decomposition(matrix):
    n, m = matrix.shape
    P = np.identity(n)
    L = np.identity(n)
    U = matrix.copy()
    PF = np.identity(n)
    LF = np.zeros((n,n))

    for k in range(0, n - 1):
        index = np.argmax(abs(U[k:,k]))
        index = index + k

        # Pivoting
        if index != k:
            P = np.identity(n)
            P[[index,k],k:n] = P[[k,index],k:n]
            U[[index,k],k:n] = U[[k,index],k:n]
            PF = np.dot(P,PF)
            LF = np.dot(P,LF)

        # Decomposition
        L = np.identity(n)
        for j in range(k+1,n):
            L[j,k] = -(U[j,k] / U[k,k])
            LF[j,k] = (U[j,k] / U[k,k])
        U = np.dot(L,U)

    np.fill_diagonal(LF, 1)
    return PF, LF, U

def solve_with_lu(A, b):
    # Get matrices from decomposition
    PF, LF, U = LU_partial_decomposition(A)

    #  L * y = P * b
    Pb = np.dot(PF, b)
    y = scipy.linalg.solve_triangular(LF, Pb, lower=True)

    #  U * x = y
    x = scipy.linalg.solve_triangular(U, y)
    return x

# Lecture system parameters
A_lecture = np.array([[1, 2, 1], [1, -2, 2], [2, 12, -2]], dtype=float)
b_lecture = np.array([0, 4, 4], dtype=float)

solution = solve_with_lu(A_lecture, b_lecture)
print("Solution to the extended LU decomposition problem:")
print(solution)

Solution to the extended LU decomposition problem:
[11.  -2.5 -6. ]
